mount drive

In [ ]:
!git clone https://github.com/THU-KEG/AgentIF.git
%cd AgentIF
!ls

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

install libraries

In [ ]:
! pip install -q openai
! pip install -q torch
! pip install -q --upgrade transformers
! pip install -q datasets
! pip install -q langgraph
! pip install -q langchain
! pip install -q langchain-openai

path to the AgentIF data

In [ ]:
AGENTIF_FILE_PATH = "/content/drive/MyDrive/TokenReductionProject/Data/eval_criteria_with_attention_scores_probe_Answer: _model_Qwen2.5-7B_DOWNSAMPLED.pkl"

set up environment

In [ ]:
import os

# env variables
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")  # add under Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # add under Colab Secrets
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # restricts visibility of gpus to only the first gpu (gpu at index 0)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


imports

In [ ]:
import warnings

# remove warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
from copy import deepcopy
from tqdm.notebook import tqdm
from typing import TypedDict
import asyncio
import random
import re
import time

import gc

set up device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"working on {device}")

Hugging Face login

In [ ]:
!pip install huggingface_hub
from huggingface_hub import login, whoami

In [ ]:
hf_token = os.environ["HF_TOKEN"]  # set before running, e.g. via Colab Secrets
login(token=hf_token)

In [ ]:
user = whoami()
print(f"Logged in as {user['name']}")

aux model to get attention scores

In [ ]:
# ---------- 0) Load aux model once ----------
AUX_MODEL = "Qwen/Qwen2.5-7B"

_aux_tok = AutoTokenizer.from_pretrained(AUX_MODEL, use_fast=True)
if _aux_tok.pad_token is None:
    _aux_tok.pad_token = _aux_tok.eos_token

_aux_model = AutoModelForCausalLM.from_pretrained(
    AUX_MODEL,
    dtype=torch.bfloat16,
    device_map="auto" if torch.cuda.is_available() else None,
    attn_implementation="eager"
).eval()

_aux_model.config.use_cache = False

In [ ]:
def _sent_spans(text: str):
    # returns [(char_start, char_end, span_text), ...]
    spans = []
    for m in re.finditer(r".+?(?:[.!?](?:\s+|$)|\n{2,}|$)", text, flags=re.S):
        s = m.group(0)
        if s.strip():
            spans.append((m.start(), m.end(), s))
    return spans

def _approx_tokens(s: str) -> int:
    return max(1, len(s) // 4)  # cheap heuristic; good enough for budgeting

@torch.no_grad()
def get_attention_scores(context, objective, *, top_layers=8, max_seq_len=2048, ctx_stride=256):
    context = context.strip()
    objective = objective.strip()
    if not context:
        return []

    # 1) tokenize objective once
    enc_obj = _aux_tok(objective, add_special_tokens=False, return_tensors="pt")
    obj_ids = enc_obj["input_ids"][0]

    # 2) tokenize context once (with offsets)
    enc_ctx = _aux_tok(context, add_special_tokens=False, return_tensors="pt",
                       return_offsets_mapping=True)
    ctx_ids = enc_ctx["input_ids"][0]
    ctx_offsets = enc_ctx["offset_mapping"][0].tolist()

    # 3) sep + chunk size
    sep_ids = _aux_tok("\n\n", add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    sep_len = sep_ids.numel()
    max_ctx_chunk = max(32, max_seq_len - (obj_ids.numel() + sep_len + 2))

    device = _aux_model.device

    # 4) CPU accumulator
    imp_cpu = torch.zeros((ctx_ids.numel(),), dtype=torch.float32)

    start = 0
    while start < ctx_ids.numel():
        end = min(ctx_ids.numel(), start + max_ctx_chunk)
        chunk_ids = ctx_ids[start:end]

        # 5a) build input: chunk + sep + objective
        input_ids = torch.cat([chunk_ids, sep_ids, obj_ids], dim=0).unsqueeze(0).to(device)
        attn_mask = torch.ones_like(input_ids)

        ctx_len_with_sep = chunk_ids.numel() + sep_len
        seq_len = input_ids.shape[1]
        q_pos = list(range(ctx_len_with_sep, seq_len))  # objective token positions

        # 5b) forward (big memory happens here, but seq_len is capped)
        out = _aux_model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True,
            use_cache=False,
            return_dict=False,
        )
        # depending on model, attentions are out[-1] or out[2]; safest:
        attns = out[-1]

        # 5c) compute importance for this chunk (on GPU, small)
        L = len(attns)
        l0 = max(0, L - top_layers)
        imp_chunk = torch.zeros((ctx_len_with_sep,), device=device)

        for l in range(l0, L):
            a = attns[l][0]  # (heads, seq, seq)
            imp_chunk += a[:, q_pos, :ctx_len_with_sep].sum(dim=(0, 1))

        # 5d) drop sep tokens, move to CPU, add into global accumulator
        imp_ctx_only_cpu = imp_chunk[:chunk_ids.numel()].float().cpu()
        imp_cpu[start:end] += imp_ctx_only_cpu

        # 5e) cleanup to avoid accumulation across chunks
        del out, attns, imp_chunk, imp_ctx_only_cpu, input_ids, attn_mask, a
        torch.cuda.empty_cache()
        gc.collect()

        if end == ctx_ids.numel():
            break
        start = end - ctx_stride  # overlap

    # 6) normalize once
    imp_cpu = imp_cpu / (imp_cpu.sum() + 1e-9)

    # 7) token scores -> sentence/span scores
    spans = _sent_spans(context)  # returns (char_s, char_e, text)
    # map token scores to spans via offsets overlap
    scored = []
    for i, (sp_s, sp_e, sp_txt) in enumerate(spans):
        s = 0.0
        for tok_i, (ts, te) in enumerate(ctx_offsets):
            if te <= sp_s:
                continue
            if ts >= sp_e:
                break
            if te > ts and ts < sp_e and te > sp_s:
                s += float(imp_cpu[tok_i].item())
        scored.append((i, s, sp_txt))

    return scored

def compress_with_attention_scores(
    scored_spans: list,
    *,
    token_budget: int = 600,
    keep_first_span: bool = True, # keeps Goal/system-ish preface
    ) -> str:

    """
    Given attention scores for each span, keeps as many sentences as possible, adding on the highest attn sentences until token budget is met
    """

    # Select spans under budget
    keep = set()
    if keep_first_span:
        keep.add(0)

    used = sum(_approx_tokens(scored_spans[i][2]) for i in keep) if keep else 0

    for i, score, txt in sorted(scored_spans, key=lambda x: x[1], reverse=True):
        if i in keep:
            continue
        t = _approx_tokens(txt)
        if used + t <= token_budget:
            keep.add(i)
            used += t

    return "\n".join(scored_spans[i][2].strip() for i in sorted(keep))

def compress_text(
    context: str,
    objective: str,
    *,
    token_budget: int = 600,
    top_layers: int = 8,          # use last N layers for "more semantic" attention
    keep_first_span: bool = True, # keeps Goal/system-ish preface
):
  scored = get_attention_scores(context, objective, top_layers=top_layers)
  compressed = compress_with_attention_scores(scored,token_budget=token_budget,keep_first_span=keep_first_span)
  return compressed

agent code


In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
class State(TypedDict, total=False):
    goal: str
    plan: str
    research: str
    draft: str
    final: str
    critique: str

In [ ]:
class TooMany429s(Exception):
    pass

async def abatch_retry(
    llm, prompts, *, config=None,
    safety_ms=250,
    timeout_s=120,
    max_total_wait_s=60.0,   # prevents “stuck all night” on one chunk
):
    start = time.monotonic()

    while True:
        try:
            result = await asyncio.wait_for(
                llm.abatch(prompts, config=config),
                timeout=timeout_s
            )
            return result

        except asyncio.TimeoutError:
            # transient stall; short backoff
            await asyncio.sleep(1.0 + random.uniform(0, 0.2))

        except Exception as e:
            msg = (str(e) or "").lower()
            is_429 = ("429" in msg) or ("rate limit" in msg) or ("ratelimit" in msg) or ("tpm" in msg)
            if not is_429:
                raise

            if time.monotonic() - start > max_total_wait_s:
                raise TooMany429s(f"429 loop > {max_total_wait_s}s for batch size {len(prompts)}") from e

            # server-guided sleep
            delay_s = None
            m = re.search(r"try again in (\d+)\s*ms", msg)
            if m:
                delay_s = int(m.group(1)) / 1000.0
            else:
                m = re.search(r"try again in ([0-9.]+)\s*s", msg)
                if m:
                    delay_s = float(m.group(1))
            if delay_s is None:
                delay_s = 1.0

            delay_s += safety_ms / 1000.0
            delay_s += random.uniform(0, 0.05)

            await asyncio.sleep(delay_s)

In [ ]:
#numbers derived from the median of lengths for each node
# _approx_tokens = characters / 4
init_tok_lim = 859
planner_tok_lim = 323
research_tok_lim = 150
builder_tok_lim = 370
critic_tok_lim = 516

max_concurrency = 6

async def abatch_in_chunks(
    llm, prompts, *,
    chunk_size=16,
    min_chunk_size=1,
    config=None
):
    out = []
    i = 0
    cur_chunk = min_chunk_size  # start conservative; ramp up

    success_streak = 0

    while i < len(prompts):
        size = min(cur_chunk, len(prompts) - i)
        chunk = prompts[i:i+size]

        try:
            out.extend(await abatch_retry(llm, chunk, config=config))
            i += size
            success_streak += 1

            # ramp up slowly after a few successes
            if success_streak >= 3 and cur_chunk < chunk_size:
                cur_chunk = min(chunk_size, cur_chunk * 2)
                success_streak = 0

        except TooMany429s:
            success_streak = 0
            # back off quickly
            cur_chunk = max(min_chunk_size, cur_chunk // 2)
            await asyncio.sleep(1.0)  # small pause to let TPM window breathe

    return out

async def planner_node(states: list[State]) -> list[State]: #no compression done here bc this round of compression is done outside
    prompts = [
        f"Goal: {s['goal']}\n"
        "Create a numbered plan (3-6 steps) with concrete subtasks."
        for s in states
    ]
    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    out_states = []
    for s, msg in zip(states, msgs):
        s2 = dict(s)
        s2["plan_raw"] = msg.content
        out_states.append(s2)

    return out_states

async def researcher_node(states: list[State]) -> list[State]:
    INSTR = "List key facts/constraints/assumptions. Keep it short."
    PROBE = "Research:"

    prompts = []
    out_states = []

    for s in states:
        s2 = dict(s)
        s2["plan_raw"] = s.get("plan_raw","")
        s2["plan_comp"] = compress_text(s2.get('plan_raw',''),PROBE,token_budget=planner_tok_lim) #compress the output of the planning node

        context = f"Goal: {s2['goal']}\nPlan:\n{s2.get('plan_comp','')}"

        prompts.append(context + "\n\n" + INSTR + "\n\n" + PROBE)
        out_states.append(s2)

    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    for s, msg in zip(out_states, msgs):
        s["research_raw"] = msg.content

    return out_states

async def builder_node(states: list[State]) -> list[State]:
    INSTR = "Write the deliverable now."
    PROBE = "Deliverable:"

    prompts = []
    out_states = []
    for s in states:
        s2 = dict(s)
        s2["research_raw"] = s.get("research_raw","")
        s2["research_comp"] = compress_text(s2.get('research_raw',''),PROBE,token_budget=research_tok_lim) #compress the output of the research node

        context = (
            f"Goal: {s2['goal']}\n"
            f"Plan:\n{s2.get('plan_comp','')}\n"
            f"Research:\n{s2.get('research_comp','')}"
        )
        prompts.append(context + "\n\n" + INSTR + "\n\n" + PROBE)

        out_states.append(s2)

    msgs = await abatch_in_chunks(llm, prompts, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    for s, msg in zip(out_states, msgs):
        s["draft_raw"] = msg.content

    return out_states

async def critic_node(states: list[State]) -> list[State]:
    INSTR_CRIT = "Critique the draft with bullet points (issues + fixes)."
    INSTR_FINAL  = "Output a revised FINAL version."
    PROBE_CRIT = "Critique:"
    PROBE_FINAL = "Response:"

    out_states = []
    prompts1 = []
    for s in states:
        s2 = dict(s)
        s2["draft_comp"] = compress_text(s2.get('draft_raw',''),PROBE_CRIT,token_budget=builder_tok_lim) #compress the output of the builder node
        context = f"Goal: {s2['goal']}\nDraft:\n{s2.get('draft_comp','')}"
        prompts1.append(context + "\n\n" + INSTR_CRIT + "\n\n" + PROBE_CRIT)

        out_states.append(s2)

    msgs1 = await abatch_in_chunks(llm, prompts1, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    for s2, msg1 in zip(out_states, msgs1):
        s2["critique_raw"] = msg1.content

    prompts2 = []
    for s2 in out_states:
        s2["critique_comp"] = compress_text(s2.get("critique_raw", ""), PROBE_FINAL,token_budget=critic_tok_lim) #compress the output of the critic node

        context = f"Goal: {s2['goal']}\nDraft:\n{s2.get('draft_comp','')}\n\nCritique:\n{s2.get('critique_comp','')}"
        prompts2.append(context + "\n\n" + INSTR_FINAL + "\n\n" + PROBE_FINAL)

    msgs2 = await abatch_in_chunks(llm, prompts2, config={"max_concurrency": max_concurrency}) #reduce if hitting rate limits, increase if not using all resources

    for s2, msg2 in zip(out_states, msgs2):
        s2["final"] = msg2.content

    return out_states

In [ ]:
async def run_agentic_system(prompts: list[str]) -> tuple[list[str],list[str],list[str]]: #returns full output list + final output list

    states = [{"goal": p} for p in prompts]
    states = await planner_node(states)
    states = await researcher_node(states)
    states = await builder_node(states)
    states = await critic_node(states)

    raw_outputs = []
    full_outputs = []
    outputs = []
    for s in states:
        raw_output = {}

        raw_output["plan"] = s.get('plan_raw','')
        raw_output["research"] = s.get('research_raw','')
        raw_output["draft"] = s.get('draft_raw','')
        raw_output["critique"] = s.get('critique_raw','')
        raw_output["final"] = s.get('final','')

        raw_outputs.append(raw_output)

        full_output = {}

        full_output["plan"] = s.get('plan_comp','')
        full_output["research"] = s.get('research_comp','')
        full_output["draft"] = s.get('draft_comp','')
        full_output["critique"] = s.get('critique_comp','')
        full_output["final"] = s.get('final','')

        full_outputs.append(full_output)
        outputs.append(s.get("final", ""))

    return raw_outputs, full_outputs, outputs

load the AgentIF data

In [ ]:
p = Path(AGENTIF_FILE_PATH)

def load_agentif_rows(path: Path):
    if path.suffix == ".pkl":
        obj = pd.read_pickle(path)  # could be DataFrame or list[dict]
        if isinstance(obj, pd.DataFrame):
            return obj.to_dict("records")
        return obj
    elif path.suffix == ".json":
        return json.loads(path.read_text())
    elif path.suffix in [".jsonl", ".jl"]:
        return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

rows = load_agentif_rows(p)
print("Loaded rows:", len(rows))
print("Example rows:", rows[1])
print("Example keys:", rows[1].keys())
print("Example values:", rows[1].values())

run the agentic system on every instruction

In [ ]:
BATCH_SIZE = 32

out_rows = []
batch = []
for r in tqdm(rows): #runs agentic system on all of the outputs
    rr = deepcopy(r)
    scored_spans = rr["scored_spans"]
    full_input = compress_with_attention_scores(scored_spans, token_budget = init_tok_lim)

    batch.append([full_input, rr])

    #runs whole thing every [batch size] prompts
    if (len(batch) == BATCH_SIZE):
      batched_prompts = [this_rr[0] for this_rr in batch]
      batched_rr = [this_rr[1] for this_rr in batch]
      raw_outputs, full_outputs, outputs = await run_agentic_system(batched_prompts) #use await to run other things simultaneously
      for row, raw_output, full_output, output in zip(batched_rr, raw_outputs, full_outputs, outputs):
        row["output"] = {"role": "assistant", "content": output, "total_content":full_output, "raw_total_content":raw_output} #raw total content is the uncompressed agentic yap, total content is the total but compressed agentic yap, content is just the final
      out_rows.extend(batched_rr)
      batch = []

#final partial batch
if (len(batch) > 0):
  batched_prompts = [this_rr[0] for this_rr in batch]
  batched_rr = [this_rr[1] for this_rr in batch]
  raw_outputs, full_outputs, outputs = await run_agentic_system(batched_prompts) #use await to run other things simultaneously
  for row, raw_output, full_output, output in zip(batched_rr, raw_outputs, full_outputs, outputs):
      row["output"] = {"role": "assistant", "content": output, "total_content":full_output, "raw_total_content":raw_output} #raw total content is the uncompressed agentic yap, total content is the total but compressed agentic yap, content is just the final
  out_rows.extend(batched_rr)
  batch = []

In [ ]:
def convert_numpy(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

In [ ]:
EXPERIMENT_NUM = "1"
AGENT_OUTPUT_PATH_JSON = Path("/content/drive/MyDrive/TokenReductionProject/Results/AgentOutputs/agent_outputs_for_eval_" + EXPERIMENT_NUM +".json")
AGENT_OUTPUT_PATH_JSON.write_text(json.dumps(out_rows, default=convert_numpy, ensure_ascii=False, indent=2))
print("Wrote", AGENT_OUTPUT_PATH_JSON, "with rows:", len(out_rows))

In [ ]:
AGENT_OUTPUT_PATH_PKL = Path("/content/drive/MyDrive/TokenReductionProject/Results/AgentOutputs/agent_outputs_for_eval_" + EXPERIMENT_NUM +".pkl")

pkl_df = pd.DataFrame(out_rows)
pkl_df.to_pickle(AGENT_OUTPUT_PATH_PKL)

print("Wrote", AGENT_OUTPUT_PATH_PKL, "with rows:", len(out_rows))